# Stellar density with E(B-V) contours (rubin_sim MAF)

This notebook evaluates `maf.StarDensityMetric` on a full-sky `HealpixSlicer`, overlays E(B-V) contours from `maf.DustMap`, shades the region that passes a joint cut on both, and outlines the Rubin baseline survey footprint and its Wide Fast Deep (WFD) area.

- **Footprint:** `rubin_scheduler.scheduler.utils.get_current_footprint`. This is the scheduler's own region map behind the [baseline WFD](https://survey-strategy.lsst.io/baseline/wfd.html) (about 19.6k deg²). WFD is taken as the same set of regions that MAF's `make_wfd_subset_slicer` uses.

- **Stellar density:** `maf.TrilegalDensityMap` (TRILEGAL, includes the bulge) or `maf.StellarDensityMap` (CatSim/GalFast; disk and halo only, **no bulge**).
- **Dust:** `maf.DustMap` (SFD E(B-V)).

Neither map depends on visits, so no OpSim database is needed. The maps are loaded straight onto the slicer's slice points.

## 1. Environment check

Checks that `rubin_sim` imports and that the map files are where MAF looks for them (`$RUBIN_SIM_DATA_DIR`, or `~/rubin_sim_data` if that isn't set).

If the maps are missing, either point `RUBIN_SIM_DATA_DIR` at an existing copy, or run this in a terminal and then restart the kernel:

```bash
pip install --user rubin-sim
rs_download_data --dirs maps
scheduler_download_data      # footprint inputs: scheduler/dust_maps, Euclid contour
```

In [ ]:
import os
# os.environ["RUBIN_SIM_DATA_DIR"] = "/path/to/rubin_sim_data"   # uncomment to override

import rubin_sim
from rubin_scheduler.data import get_data_dir

print("rubin_sim", rubin_sim.__version__)
data_dir = get_data_dir()
print("data dir:", data_dir)
for sub in ["maps/TriMaps", "maps/StarMaps", "maps/DustMaps", "scheduler/dust_maps"]:
    p = os.path.join(data_dir, sub)
    print(f"  {sub:20s}", "OK" if os.path.isdir(p) else "MISSING")

In [ ]:
import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
import rubin_sim.maf as maf

%matplotlib inline

## 2. Settings

In [ ]:
NSIDE = 64               # native nside of the star-density maps
FILTER = "r"             # u g r i z y
STAR_MODEL = "trilegal"  # "trilegal" (has bulge) or "galfast" (no bulge)
MAG_LIMIT = 17.0         # count stars brighter than this (valid ~15-28)
EBV_LEVELS = [0.12]
COORD = "C"              # plot in equatorial (celestial); use ["C", "G"] for Galactic

# Joint cut: pixels passing BOTH conditions are shaded on the map
STARDENS_CUT = 0.0002    # stars / arcsec^2 (same units as the metric)
STARDENS_SIDE = "below"  # keep pixels "above" or "below" STARDENS_CUT
EBV_CUT = 0.12           # E(B-V) in mag
EBV_SIDE = "below"       # keep pixels "above" or "below" EBV_CUT
CUT_COLOR = "#ff3b8b"

# Rubin baseline footprint (labels from rubin_scheduler's current footprint).
# Default WFD = the regions MAF's make_wfd_subset_slicer uses
WFD_LABELS = ["lowdust", "euclid_overlap", "virgo", "bulgy", "LMC_SMC"]
RESTRICT_CUT_TO_WFD = True  # joint cut counts only pixels inside the WFD
SHOW_FULL_FOOTPRINT = True  # also outline the whole survey footprint (dashed)
WFD_COLOR = "#ff9f1c"
OUTFILE = f"stardens_{STAR_MODEL}_{FILTER}{MAG_LIMIT:g}_ebv.png"

## 3. Load the maps onto the slice points

In [ ]:
slicer = maf.HealpixSlicer(nside=NSIDE, verbose=False)
sp = dict(slicer.slice_points)  # ra, dec (rad), sid, nside, gall, galb

if STAR_MODEL == "trilegal":
    maf.TrilegalDensityMap(filtername=FILTER, nside=NSIDE).run(sp)
else:
    maf.StellarDensityMap(filtername=FILTER).run(sp)
maf.DustMap(nside=NSIDE).run(sp)

print(sorted(k for k in sp if k.startswith(("star", "ebv"))))

## 4. Evaluate `StarDensityMetric` at each pixel

This gives stars per square arcsecond brighter than `MAG_LIMIT`.

In [ ]:
metric = maf.StarDensityMetric(filtername=FILTER, mag_limit=MAG_LIMIT)
lf_key, bin_key = f"starLumFunc_{FILTER}", f"starMapBins_{FILTER}"
stardens = np.full(hp.nside2npix(NSIDE), hp.UNSEEN)
for i in range(stardens.size):
    val = metric.run(None, slice_point={lf_key: sp[lf_key][i], bin_key: sp[bin_key]})
    if val != metric.badval and val > 0:
        stardens[i] = val

ebv = np.asarray(sp["ebv"], dtype=float)
logdens = np.where(stardens > 0, np.log10(stardens), hp.UNSEEN)
print(f"valid pixels: {(stardens > 0).sum()} / {stardens.size}")

## 5. Check coordinate frames and rotate

MAF assumes both maps are stored in equatorial coordinates. This cell checks that from the data: stars and dust both pile up at low |b|, so it tests whether the brightest 5% of pixels sit closer to the Galactic plane when the map is read as equatorial (C) or as Galactic (G). Each map is then rotated into the plotting frame separately.

In [ ]:
def stored_frame(m):
    """Guess whether a full-sky healpix map is stored in equatorial ("C") or
    Galactic ("G") pixel order. Both stars and dust pile up at low |b|, so
    take the brightest 5% of pixels and ask which interpretation puts them
    closer to the Galactic plane."""
    pix = np.arange(m.size)
    good = np.isfinite(m) & (m != hp.UNSEEN)
    top = good & (m >= np.percentile(m[good], 95))
    theta, phi = hp.pix2ang(NSIDE, pix[top])
    b_if_G = 90.0 - np.degrees(theta)                       # pixel lat is b
    b_if_C = 90.0 - np.degrees(hp.Rotator(coord=["C", "G"])(theta, phi)[0])
    return "C" if np.median(np.abs(b_if_C)) < np.median(np.abs(b_if_G)) else "G"


def to_frame(m, src, dst):
    """Rotate a healpix map from frame src to dst, preserving UNSEEN pixels."""
    if src == dst:
        return m
    rot = hp.Rotator(coord=[src, dst])
    bad = rot.rotate_map_pixel((m == hp.UNSEEN).astype(float)) > 0.5
    out = rot.rotate_map_pixel(np.where(m == hp.UNSEEN, 0.0, m))
    out[bad] = hp.UNSEEN
    return out


target = COORD[-1]
star_frame, dust_frame = stored_frame(logdens), stored_frame(ebv)
print(f"stellar density stored in {star_frame}, E(B-V) stored in {dust_frame}")
logdens_p = to_frame(logdens, star_frame, target)
ebv_p = to_frame(ebv, dust_frame, target)
frame = {"C": "Equatorial", "G": "Galactic"}[target]

## 6. Rubin survey footprint and WFD

Loads the scheduler's current footprint. Every pixel carries a region label (`lowdust`, `euclid_overlap`, `virgo`, `bulgy`, `LMC_SMC`, `nes`, `dusty_plane`, `scp`, or empty if outside the survey). The WFD is the union of `WFD_LABELS`.

For reference, the low-dust WFD in that code is roughly: smoothed SFD E(B-V) < 0.199, −70° < Dec < +15° (upper limit lowered by 12° where b < 0), plus the Euclid-overlap extension. The WFD-level "bulgy" Galactic plane/bulge is a set of circles, the largest centred on (RA, Dec) = (266.3°, −29°) with radius 17°. Check the code for the exact current values; the footprint map is in equatorial coordinates.

In [ ]:
from rubin_scheduler.scheduler.utils import get_current_footprint

_, fp_labels = get_current_footprint(nside=NSIDE)
pix_area = hp.nside2pixarea(NSIDE, degrees=True)

labels, counts = np.unique(fp_labels[fp_labels != ""], return_counts=True)
for lab, n in zip(labels, counts):
    print(f"  {lab:15s} {n * pix_area:8,.0f} deg^2" + ("  (WFD)" if lab in WFD_LABELS else ""))

wfd = np.isin(fp_labels, WFD_LABELS)
survey = fp_labels != ""
print(f"WFD total: {wfd.sum() * pix_area:,.0f} deg^2;  full footprint: {survey.sum() * pix_area:,.0f} deg^2")

# Footprint is equatorial; rotate into the plotting frame like the other maps
wfd_p = to_frame(wfd.astype(float), "C", target) > 0.5
survey_p = to_frame(survey.astype(float), "C", target) > 0.5

## 7. Joint cut on stellar density and E(B-V)

Selects the pixels that pass both the stellar-density cut and the E(B-V) cut set in **Settings**, and reports their sky area. With `RESTRICT_CUT_TO_WFD = True`, only pixels inside the WFD are kept. The selection is made on the maps after they've been rotated into the plotting frame, so it lines up with the plot.

In [ ]:
def side_mask(m, cut, side):
    good = np.isfinite(m) & (m != hp.UNSEEN)
    if side == "above":
        return good & (m >= cut)
    if side == "below":
        return good & (m <= cut)
    raise ValueError(f"side must be 'above' or 'below', got {side!r}")


# logdens_p is log10(stars/arcsec^2), so compare against log10 of the cut
sel_all = side_mask(logdens_p, np.log10(STARDENS_CUT), STARDENS_SIDE) & side_mask(ebv_p, EBV_CUT, EBV_SIDE)
sel = sel_all & wfd_p if RESTRICT_CUT_TO_WFD else sel_all

sym = {"above": ">=", "below": "<="}
cut_label = (f"n* {sym[STARDENS_SIDE]} {STARDENS_CUT:g} arcsec$^{{-2}}$ & "
             f"E(B-V) {sym[EBV_SIDE]} {EBV_CUT:g}")
if RESTRICT_CUT_TO_WFD:
    cut_label += " (in WFD)"
print(f"all sky:  {sel_all.sum() * pix_area:,.0f} deg^2 pass the cut")
print(f"in WFD:   {(sel_all & wfd_p).sum() * pix_area:,.0f} deg^2 "
      f"({100 * (sel_all & wfd_p).sum() / max(wfd_p.sum(), 1):.1f}% of WFD)")
print(f"{cut_label.replace('$^{-2}$', '^-2')}: {sel.sum()} pixels = {sel.sum() * pix_area:,.0f} deg^2 "
      f"({100 * sel.mean():.1f}% of sky)")

## 8. Plot stellar density, E(B-V) contours, joint cut and footprint

In [ ]:
fig = plt.figure(figsize=(11, 6.5))
hp.mollview(
    logdens_p, fig=fig.number, cmap="viridis",
    title=f"Stellar density ({FILTER} < {MAG_LIMIT:g}, {STAR_MODEL}) with E(B-V) contours [{frame}]",
    unit=r"log$_{10}$ stars arcsec$^{-2}$",
)
hp.graticule(dpar=30, dmer=60, alpha=0.4)

# Project E(B-V) onto the same Mollweide grid and contour it on the same axes
proj = hp.projector.MollweideProj(xsize=1600)
img = proj.projmap(ebv_p, lambda x, y, z: hp.vec2pix(NSIDE, x, y, z))
img = np.ma.masked_where(~np.isfinite(img) | (img == hp.UNSEEN), img)
x0, x1, y0, y1 = proj.get_extent()
X, Y = np.meshgrid(np.linspace(x0, x1, img.shape[1]), np.linspace(y0, y1, img.shape[0]))

ax = plt.gca()
cs = ax.contour(X, Y, img, levels=EBV_LEVELS, colors="white", linewidths=0.8)
ax.clabel(cs, fmt="%.2f", fontsize=7)

# Joint-cut layer: translucent fill plus a solid outline
selimg = proj.projmap(sel.astype(float), lambda x, y, z: hp.vec2pix(NSIDE, x, y, z))
selimg = np.ma.masked_where(~np.isfinite(selimg) | (selimg == hp.UNSEEN), selimg)
ax.contourf(X, Y, selimg, levels=[0.5, 1.5], colors=[CUT_COLOR], alpha=0.35)
ax.contour(X, Y, selimg, levels=[0.5], colors=[CUT_COLOR], linewidths=1.2)

# Footprint outlines: WFD solid, full survey footprint dashed
def outline(mask, **kw):
    im = proj.projmap(mask.astype(float), lambda x, y, z: hp.vec2pix(NSIDE, x, y, z))
    im = np.ma.masked_where(~np.isfinite(im) | (im == hp.UNSEEN), im)
    plt.gca().contour(X, Y, im, levels=[0.5], **kw)

outline(wfd_p, colors=[WFD_COLOR], linewidths=1.8)
if SHOW_FULL_FOOTPRINT:
    outline(survey_p, colors=["0.85"], linewidths=1.0, linestyles="--")

from matplotlib.patches import Patch
from matplotlib.lines import Line2D
handles = [Patch(facecolor=CUT_COLOR, edgecolor=CUT_COLOR, alpha=0.5, label=cut_label),
           Line2D([], [], color="white", lw=0.8, label="E(B-V) contours"),
           Line2D([], [], color=WFD_COLOR, lw=1.8, label="WFD footprint")]
if SHOW_FULL_FOOTPRINT:
    handles.append(Line2D([], [], color="0.85", lw=1.0, ls="--", label="full survey footprint"))
ax.legend(
    handles=handles,
    loc="lower left", bbox_to_anchor=(0.0, -0.02), fontsize=8, framealpha=0.8,
    facecolor="0.3", labelcolor="white",
)

fig.savefig(OUTFILE, dpi=150, bbox_inches="tight")
print("wrote", OUTFILE)
plt.show()

## 9. Compare with an external HEALPix map

Reads an external HEALPix map, by default `rubin_wfd_stellar_masked_nside64.fits` (column `wfd_usable_fraction`, the usable fraction of each WFD pixel). It is compared with the maps above in four ways:

1. **Footprint:** area covered by the external map vs our WFD.
2. **Drivers:** the external value against stellar density and E(B-V), with Spearman rank correlations.
3. **Cut agreement:** our joint cut (section 7) against the external map thresholded at `EXT_MIN`, as a confusion table and an agreement map.
4. **Effective area:** our cut area weighted by the external fraction.

All statistics are computed in equatorial pixels (the native frame of the footprint and the external map). The maps are rotated only for display. The file's `COORDSYS` and `NSIDE` header keywords are read and honoured.

In [ ]:
from astropy.io import fits
from scipy.stats import spearmanr

EXT_MAP_FILE = "rubin_wfd_stellar_masked_nside64.fits"
EXT_FIELD = 0            # column index (or name) to read
EXT_LABEL = "WFD usable fraction"
EXT_MIN = 0.5            # threshold that counts a pixel as "good" in the external map

hdr = fits.getheader(EXT_MAP_FILE, 1)
ext_coord = hdr.get("COORDSYS", "C").strip()[0].upper()
ext_coord = {"E": "E", "G": "G"}.get(ext_coord, "C")   # 'C'/'Q' -> equatorial
ext = hp.read_map(EXT_MAP_FILE, field=EXT_FIELD)       # always returned in RING order
if hp.get_nside(ext) != NSIDE:
    ext = hp.ud_grade(ext, NSIDE)
print(f"{EXT_MAP_FILE}: nside={hdr.get('NSIDE')}, ordering={hdr.get('ORDERING')}, coordsys={ext_coord}")


def to_frame_nearest(m, src, dst):
    """Rotate by nearest-pixel lookup (no interpolation): good for masks and labels."""
    if src == dst:
        return m
    th, ph = hp.pix2ang(NSIDE, np.arange(m.size))
    th2, ph2 = hp.Rotator(coord=[dst, src])(th, ph)
    return m[hp.ang2pix(NSIDE, th2, ph2)]


# Everything in equatorial pixels for the statistics
ext_c = to_frame_nearest(ext, ext_coord, "C")
ext_ok = np.isfinite(ext_c) & (ext_c != hp.UNSEEN)
logdens_c = to_frame_nearest(logdens, star_frame, "C")
ebv_c = to_frame_nearest(ebv, dust_frame, "C")
sel_all_c = (side_mask(logdens_c, np.log10(STARDENS_CUT), STARDENS_SIDE)
             & side_mask(ebv_c, EBV_CUT, EBV_SIDE))
sel_c = sel_all_c & wfd if RESTRICT_CUT_TO_WFD else sel_all_c

print(f"external map covers {ext_ok.sum() * pix_area:,.0f} deg^2; "
      f"values {ext_c[ext_ok].min():.3g} to {ext_c[ext_ok].max():.3g}; "
      f"weighted area {ext_c[ext_ok].sum() * pix_area:,.0f} deg^2")

### 9a. Footprint: external coverage vs our WFD

In [ ]:
both = ext_ok & wfd
print(f"in both:            {both.sum() * pix_area:8,.0f} deg^2")
print(f"external only:      {(ext_ok & ~wfd).sum() * pix_area:8,.0f} deg^2")
print(f"our WFD only:       {(~ext_ok & wfd).sum() * pix_area:8,.0f} deg^2")
print(f"Jaccard overlap:    {both.sum() / max((ext_ok | wfd).sum(), 1):.3f}")

# External-only pixels by footprint label: shows which regions differ
lab, cnt = np.unique(fp_labels[ext_ok & ~wfd], return_counts=True)
for l, n in zip(lab, cnt):
    print(f"  external-only in {l or '(outside footprint)':22s} {n * pix_area:7,.0f} deg^2")

### 9b. External map vs stellar density and E(B-V)

In [ ]:
good = ext_ok & (logdens_c != hp.UNSEEN) & np.isfinite(ebv_c) & (ebv_c > 0)
x1, x2, y = logdens_c[good], np.log10(ebv_c[good]), ext_c[good]
r1, r2 = spearmanr(x1, y).statistic, spearmanr(x2, y).statistic

fig2, axs = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for a, x, xl, r, cut, side in [
    (axs[0], x1, r"log$_{10}$ stars arcsec$^{-2}$", r1, np.log10(STARDENS_CUT), STARDENS_SIDE),
    (axs[1], x2, r"log$_{10}$ E(B-V)", r2, np.log10(EBV_CUT), EBV_SIDE),
]:
    hb = a.hexbin(x, y, gridsize=60, mincnt=1, bins="log", cmap="viridis")
    a.axvline(cut, color=CUT_COLOR, lw=1.5, label=f"cut ({side})")
    a.axhline(EXT_MIN, color="0.4", lw=1, ls="--", label=f"EXT_MIN = {EXT_MIN:g}")
    a.set_xlabel(xl)
    a.set_title(f"Spearman $\\rho$ = {r:+.2f}", fontsize=10)
    a.legend(fontsize=8, loc="lower left")
axs[0].set_ylabel(EXT_LABEL)
fig2.colorbar(hb, ax=axs, label="pixels (log)", shrink=0.85)
plt.show()

### 9c. Our joint cut vs the external map above `EXT_MIN`

In [ ]:
ext_good = ext_ok & (ext_c >= EXT_MIN)
a_both, a_cut, a_ext = (sel_c & ext_good), (sel_c & ~ext_good), (~sel_c & ext_good)
print(f"external >= {EXT_MIN:g}:          {ext_good.sum() * pix_area:8,.0f} deg^2")
print(f"our cut:                  {sel_c.sum() * pix_area:8,.0f} deg^2")
print(f"  both:                   {a_both.sum() * pix_area:8,.0f} deg^2")
print(f"  our cut only:           {a_cut.sum() * pix_area:8,.0f} deg^2")
print(f"  external only:          {a_ext.sum() * pix_area:8,.0f} deg^2")
print(f"Jaccard overlap:          {a_both.sum() / max((sel_c | ext_good).sum(), 1):.3f}")
print(f"mean external value inside our cut: {ext_c[sel_c & ext_ok].mean():.3f}; "
      f"outside it (in external map): {ext_c[~sel_c & ext_ok].mean():.3f}")
print(f"effective area of our cut weighted by external map: "
      f"{ext_c[sel_c & ext_ok].sum() * pix_area:,.0f} deg^2")

# Agreement map: 1 = both, 2 = our cut only, 3 = external only
agree = np.full(ext_c.size, hp.UNSEEN)
agree[a_both], agree[a_cut], agree[a_ext] = 1, 2, 3
agree_p = to_frame_nearest(agree, "C", target)
ext_p = to_frame_nearest(np.where(ext_ok, ext_c, hp.UNSEEN), "C", target)

### 9d. Maps: external map, and where it agrees with our cut

In [ ]:
from matplotlib.colors import ListedColormap

AGREE_COLORS = ["#2a9d8f", CUT_COLOR, "#e9c46a"]   # both, our cut only, external only

fig3 = plt.figure(figsize=(16, 5))
hp.mollview(ext_p, fig=fig3.number, sub=(1, 2, 1), cmap="viridis",
            title=f"{EXT_LABEL} [{frame}]", unit=EXT_LABEL)
hp.graticule(dpar=30, dmer=60, alpha=0.3)
outline(wfd_p, colors=[WFD_COLOR], linewidths=1.5)
outline(sel, colors=[CUT_COLOR], linewidths=1.0)
plt.gca().legend(handles=[Line2D([], [], color=WFD_COLOR, lw=1.5, label="our WFD"),
                          Line2D([], [], color=CUT_COLOR, lw=1.0, label="our joint cut")],
                 loc="lower left", fontsize=8)

hp.mollview(agree_p, fig=fig3.number, sub=(1, 2, 2), cmap=ListedColormap(AGREE_COLORS),
            min=0.5, max=3.5, cbar=False,
            title=f"Our cut vs {EXT_LABEL} >= {EXT_MIN:g} [{frame}]")
hp.graticule(dpar=30, dmer=60, alpha=0.3)
plt.gca().legend(handles=[Patch(color=c, label=l) for c, l in
                          zip(AGREE_COLORS, ["both", "our cut only", f"external >= {EXT_MIN:g} only"])],
                 loc="lower left", fontsize=8)
fig3.savefig(OUTFILE.replace(".png", "_vs_external.png"), dpi=150, bbox_inches="tight")
plt.show()

## 10a. TVS SC proposed Galactic-plane region

This rebuilds the region proposed in the TVS SC notebook [`GalPlaneSurvey/modified_galplane_footprint.ipynb`](https://github.com/LSST-TVSSC/software_tools/blob/main/GalPlaneSurvey/modified_galplane_footprint.ipynb), following its steps:

1. Start from the scheduler's current `bulgy` pixels.
2. Rank them by the Galactic Plane science priority map: the `combined_map` column of `priority_GalPlane_footprint_map_data_sum.fits`, divided by 6.
3. Keep the highest-priority pixels until their baseline visits reach (1 − `TVS_REDUCE_FRACTION`) of the current `bulgy` total. Visits are counted with MAF `CountMetric` and `fiveSigmaDepth > 21.5` on the baseline v5.3 OpSim run.
4. Add the Roman RGPS footprint.
5. Remove pixels at 180° < l < 270°, keep b < 11°, and add an ecliptic bridge: ecliptic longitude 235–305° and latitude ±2.5° in the barycentric mean ecliptic frame.

The region built here (`tvs_region`) feeds section 10, where `GALPLANE_MODE = "rgps_tvs"` reshapes the Galactic-plane WFD region to follow it. The TVS notebook's own output is `footprint_maps/new_rubin_gp_survey_footprint.json`, and the cell compares the rebuilt region with it.

Two differences from the original:

- **RGPS footprint:** the original reads a local `rgps_survey_healpixels.json` that isn't published. Here the RGPS wide-field boxes (`RGPS_BOXES`, defined in this cell) are used instead.
- **Visit counts:** if no OpSim database is found, the cell falls back to equal visits per pixel (`TVS_NVISITS_SOURCE = "uniform"`), so the cut becomes a 10% cut in area.

In [ ]:
import os
from astropy.io import fits

TVS_REDUCE_FRACTION = 0.10
TVS_PRIORITY_URL = ("https://raw.githubusercontent.com/LSST-TVSSC/software_tools/main/"
                    "footprint_maps/priority_GalPlane_footprint_map_data_sum.fits")
TVS_NVISITS_SOURCE = "opsim"     # "opsim" (MAF count on a baseline run) or "uniform"
TVS_OPSIM_DB = None              # path to baseline_v5.3.0_10yrs.db; None = rubin_sim get_baseline()
TVS_B_MAX = 11.0                 # keep b < this (deg)
TVS_DROP_L = (180.0, 270.0)      # drop pixels with l in this range (deg, 0-360)
TVS_BRIDGE_ELON = (235.0, 305.0) # ecliptic bridge, barycentric mean ecliptic (deg)
TVS_BRIDGE_ELAT = (-2.5, 2.5)
TVS_JSON_URL = ("https://raw.githubusercontent.com/LSST-TVSSC/software_tools/main/"
                "footprint_maps/new_rubin_gp_survey_footprint.json")

from astropy.coordinates import SkyCoord
import astropy.units as u

# RGPS wide-field boxes (Galactic l, b in deg; l in [-180, 180]), from
# roman-docs.stsci.edu/roman-community-defined-surveys/galactic-plane-survey (691.15 deg^2)
RGPS_BOXES = {
    "disk":            ((-67.0, 50.1), (-2.0, 2.0)),
    "carina_ext":      ((-79.0, -67.0), (-2.5, 2.0)),
    "bulge_bar_north": ((-10.0, 10.0), (2.0, 6.0)),
    "bulge_bar_south": ((-10.0, 10.0), (-6.0, -2.0)),
    "serpens_south":   ((26.5, 30.0), (2.0, 4.5)),
}

npix = hp.nside2npix(NSIDE)
_th, _ph = hp.pix2ang(NSIDE, np.arange(npix))
_gth, _gph = hp.Rotator(coord=["C", "G"])(_th, _ph)
gl = (np.degrees(_gph) + 180.0) % 360.0 - 180.0
gb = 90.0 - np.degrees(_gth)

# 1-2. priority map (equatorial, RING, nside 64) and the current bulgy pixels
prio = fits.getdata(TVS_PRIORITY_URL, 1, cache=True)["combined_map"] / 6.0
if prio.size != npix:
    prio = hp.ud_grade(prio, NSIDE)
gp_pix = np.where(np.asarray(fp_labels) == "bulgy")[0]
current_gp_map = np.zeros(npix)
current_gp_map[gp_pix] = prio[gp_pix]

# 3. baseline visits per pixel
nv = None
if TVS_NVISITS_SOURCE == "opsim":
    try:
        db = TVS_OPSIM_DB
        if db is None:
            from rubin_sim.data import get_baseline
            db = get_baseline()
        b_ = maf.MetricBundle(maf.CountMetric(col="observationStartMJD", metric_name="Nvisits"),
                              maf.HealpixSlicer(nside=NSIDE, use_cache=False, verbose=False),
                              "fiveSigmaDepth > 21.5", run_name="baseline")
        maf.MetricBundleGroup({"nvisits": b_}, db, out_dir="tvs_maf_out", results_db=None).run_all()
        nv = b_.metric_values.filled(0.0)
        nv_src = f"MAF Nvisits on {os.path.basename(db)}"
    except Exception as e:
        print(f"OpSim visits unavailable ({type(e).__name__}: {e}); using uniform visits per pixel")
if nv is None:
    nv = np.ones(npix)
    nv_src = "uniform (1 visit per pixel)"

current_nv = nv[gp_pix].sum()
reduced_nv = (1 - TVS_REDUCE_FRACTION) * current_nv
pixel_order = np.argsort(current_gp_map)[::-1]          # same ordering as the TVS notebook
pixel_order = pixel_order[np.isin(pixel_order, gp_pix)]
before = np.concatenate([[0.0], np.cumsum(nv[pixel_order])[:-1]])   # visits before adding each pixel
reduced = pixel_order[before <= reduced_nv]

# 4. add RGPS
rgps_core = np.zeros(npix, bool)
for _, ((l0, l1), (b0, b1)) in RGPS_BOXES.items():
    rgps_core |= (gl >= l0) & (gl < l1) & (gb >= b0) & (gb < b1)
tvs_gp = np.zeros(npix, bool)
tvs_gp[reduced] = True
tvs_gp |= rgps_core

# 5. drop l = 180-270 islands, keep b < TVS_B_MAX, add the ecliptic bridge
l360 = gl % 360.0
tvs_gp &= ~((l360 > TVS_DROP_L[0]) & (l360 < TVS_DROP_L[1]))
tvs_gp &= gb < TVS_B_MAX
elon_g, elat_g = np.meshgrid(np.linspace(*TVS_BRIDGE_ELON, 500), np.linspace(*TVS_BRIDGE_ELAT, 100))
ec = SkyCoord(lon=elon_g * u.deg, lat=elat_g * u.deg, frame="barycentricmeanecliptic").icrs
tvs_bridge = np.zeros(npix, bool)
tvs_bridge[hp.ang2pix(NSIDE, ec.ra.deg.ravel(), ec.dec.deg.ravel(), lonlat=True)] = True
tvs_region = tvs_gp | tvs_bridge

print(f"visits source: {nv_src}")
print(f"current bulgy: {gp_pix.size * pix_area:,.0f} deg^2, {current_nv:,.0f} visits -> "
      f"keep {reduced.size * pix_area:,.0f} deg^2 by priority ({100 * TVS_REDUCE_FRACTION:g}% fewer visits)")
print(f"+ RGPS, - l {TVS_DROP_L[0]:g}-{TVS_DROP_L[1]:g}, b < {TVS_B_MAX:g}: {tvs_gp.sum() * pix_area:,.0f} deg^2")
print(f"ecliptic bridge: {tvs_bridge.sum() * pix_area:,.0f} deg^2 (the TVS notebook prints 84 deg^2, "
      "but that counts rows of its sampling grid, not pixels)")
print(f"TVS region total: "
      f"{tvs_region.sum() * pix_area:,.0f} deg^2  (TVS notebook: 1,965 deg^2)")
print(f"baseline visits in TVS region: {nv[tvs_region].sum():,.0f}  (TVS notebook: 1,462,332)")

# compare with the TVS notebook's published pixel list
try:
    import json, urllib.request
    with urllib.request.urlopen(TVS_JSON_URL, timeout=30) as r:
        pub = np.zeros(npix, bool)
        pub[np.asarray(json.load(r)["new_gp_pixels"], dtype=int)] = True
    print(f"vs published JSON: both {(pub & tvs_region).sum() * pix_area:,.0f}, rebuilt only "
          f"{(tvs_region & ~pub).sum() * pix_area:,.0f}, published only {(pub & ~tvs_region).sum() * pix_area:,.0f} deg^2")
except Exception as e:
    print(f"published JSON not compared ({type(e).__name__})")

## 10. Modified footprint

Builds a new footprint from the scheduler's current one, in five steps. All of them work in equatorial pixels, the native frame of the footprint.

1. **Cut:** `lowdust` pixels that pass the joint cut stay `lowdust` (the cut is applied without the WFD restriction). Labels in `CLAIM_FROM_LABELS` that pass the cut are also turned into `lowdust`.
2. **Smooth:** the `lowdust` edge is smoothed with a Gaussian of FWHM `LOWDUST_SMOOTH_FWHM` (degrees; 0 turns it off). This uses normalised convolution over `lowdust` plus the regions it may trade with (`HOLE_FILL_LABELS`), so edges against empty sky or other WFD regions are left alone. `lowdust` can only grow into `HOLE_FILL_LABELS` pixels.
3. **Fill holes:** `lowdust` pixels lost to the cut or the smoothing are grouped into connected holes. Each hole goes to the nearest region in `HOLE_FILL_LABELS`: the one it shares the most pixel edges with, or, if it is enclosed, the one whose nearest pixel is closest. `HOLE_MODE = "pixel"` assigns each pixel separately instead.
4. **Squash bulgy islands:** connected pieces of `bulgy` smaller than `BULGY_ISLAND_MAX_AREA` deg² are given to the nearest region the same way. In the current footprint these are the two 3°-radius circles toward the anticentre, at RA, Dec ≈ (100.9°, +9.6°) and (84.9°, −5.7°).
   **Trim bulgy:** `BULGY_TRIM_FRAC` (default 10%) of the remaining `bulgy` area is then peeled off its outer edge: the pixels closest to a `HOLE_FILL_LABELS` region go first, and each goes to the nearest such region. Edges shared with `lowdust` or other WFD regions are not trimmed. This runs before the bridge, so the bridge still reaches the new edge.
5. **Bridge:** with `BRIDGE_MODE = "ecliptic"` (the default), the bridge is a band centred on the ecliptic where it crosses the Galactic plane near the bulge (RA ≈ 18h). It joins the plane region (`bulgy` or `rgps`) to `lowdust` on both the Galactic-north and Galactic-south sides. Only pixels in `BRIDGE_TAKE_FROM` (default `dusty_plane`) are relabelled, so the band stops at `lowdust`. Only band pieces that touch the plane region are kept, which excludes the anticentre crossing. The band width is solved by bisection so the bridge area equals `BRIDGE_TARGET_AREA` (default 400 deg²), to within one NSIDE-64 pixel ring. The cell reports the width, the area on each side of the plane, and whether the plane region connects to `lowdust` on each side. `BRIDGE_MODE = "box"` uses the earlier Galactic box (`BRIDGE_L`, `BRIDGE_B`) instead.

6. **Galactic-plane region:** with `GALPLANE_MODE = "rgps"`, the scheduler's `bulgy` region is replaced by the Roman Galactic Plane Survey (RGPS) wide-field footprint, grown to `RGPS_TARGET_AREA` (default 1,700 deg²), and steps 4 and 4b are skipped. The RGPS boxes are from the [Roman documentation](https://roman-docs.stsci.edu/roman-community-defined-surveys/galactic-plane-survey) and total 691.15 deg²:

   | component | l (deg) | b (deg) |
   |---|---|---|
   | disk | −67 to 50.1 | −2 to 2 |
   | Carina extension | −79 to −67 | −2.5 to 2 |
   | bulge/bar | −10 to 10 | 2 < \|b\| < 6 |
   | Serpens South | 26.5 to 30 | 2 to 4.5 |

   The time-domain and deep fields lie inside these boxes. The border is grown by adding the pixels nearest the core until the target area is reached, which makes a uniform buffer. Growth never takes `RGPS_PROTECT` regions and stays at Dec ≤ `RGPS_DEC_MAX`. Any `bulgy` left over goes to the nearest non-WFD region. The bridge (step 5) then joins the RGPS region to `lowdust`. Set `GALPLANE_MODE = "bulgy"` to go back to the trimmed scheduler `bulgy`.

   With `GALPLANE_MODE = "rgps_tvs"` (the default), the `rgps` region instead follows the TVS SC proposed region from section 10a, still at `RGPS_TARGET_AREA`. By default the TVS ecliptic-bridge pixels outside the TVS plane region are not given to `rgps`: they are reserved for the bridge (`RGPS_TVS_INCLUDE_BRIDGE = False`). Step 5 then always includes them (`BRIDGE_SEED_TVS = True`) and widens the ecliptic band around them until the bridge reaches its target area, up to `BRIDGE_MAX_WIDTH` (30°) so the bridge stays close to the ecliptic.
   - **Northern plane:** with `RGPS_NORTH_CUT = True` (the default), all of the thin northern plane at l > `RGPS_NORTH_L_MIN` is dropped. The default of 30° keeps the wide part of the TVS region (|b| out to about 6–11°, running to l ≈ 30°) and lies north of the TVS ecliptic bridge, which reaches l ≈ 28°. Beyond l ≈ 30° the TVS region narrows to a |b| ≲ 3° strip. Serpens South (l = 26.5–30°) is kept. `rgps` is not grown back (`RGPS_GROW_TO_TARGET = False`).
   - **Bridge kept out of the cut area:** the bridge may not take pixels in the removed northern plane (l > `RGPS_NORTH_L_MIN`, |b| < `BRIDGE_NORTH_PLANE_B` = 6°), so widening it doesn't refill that strip.
   - **Shortfall to the bridge:** with `BRIDGE_ABSORB_RGPS_SHORTFALL = True`, the shortfall (`RGPS_TARGET_AREA` minus the final `rgps` area) is added to the bridge target. The bridge is widened by that amount, so `rgps` + `bridge` together still total `RGPS_TARGET_AREA` + `BRIDGE_TARGET_AREA`.
   - **Too large:** pixels are removed from the northern plane first (l > `RGPS_NORTH_L_MIN`, farthest l first), then, if still needed, whatever lies farthest from the Galactic centre. This can drop the northern part of the Roman footprint.
   - **Too small:** it grows outward one pixel ring at a time, and within each ring the pixels nearest the Galactic centre are added first.

   `RGPS_PROTECT` regions are never taken. The printout reports the overlap with the TVS region and how much of the Roman wide-field footprint is kept.

**Trim the `rgps` border (step 4c-2):** `RGPS_EDGE_TRIM_FRAC` peels that fraction of the `rgps` area off its border evenly all round: the pixels closest to the edge go first, and each goes to the nearest non-WFD region. To trim to a given area instead, set `RGPS_EDGE_TRIM_TO_AREA` (deg²), which overrides the fraction. Both are off by default. The trimmed area becomes `dusty_plane`. It is not added to the bridge's target, and the bridge, the smoothing and the notch filling are not allowed to take it back. Smoothing runs after the trim.

**Smooth the `rgps` border (step 4d):** the `rgps` mask is smoothed with a Gaussian of FWHM `RGPS_SMOOTH_FWHM` (default 4°). The pixels with the highest smoothed values are kept, the same number as before, so the area doesn't change. The border can only move into non-WFD regions, never into the reserved TVS bridge or the northern area beyond `RGPS_NORTH_L_MIN`. Pixels it gives up go to the nearest non-WFD region.

**No islands (step 4e):** with `RGPS_SINGLE_PIECE = True`, only the largest connected piece of `rgps` is kept, and any detached pieces go to the nearest non-WFD region. Holes smaller than `RGPS_FILL_HOLES_MAX` enclosed entirely by `rgps` are filled in. Before that, a pixel-scale cleanup removes `rgps` pixels with fewer than `RGPS_MIN_NEIGHBOURS` (3) `rgps` neighbours, so nothing hangs on by a corner. It also fills one-pixel notches. Area lost this way counts toward the `rgps` shortfall that the bridge absorbs.

The table then reports area and pointings per component, at `POINTINGS_PER_DEG2` = 1.3 pointings per 9.6 deg².

In [ ]:
import pandas as pd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial import cKDTree

POINTINGS_PER_DEG2 = 1.3 / 9.6
NEW_FP_FILE = f"footprint_labels_cut_{STAR_MODEL}_{FILTER}{MAG_LIMIT:g}.fits"

# 1. cut
CLAIM_FROM_LABELS = []                   # other labels the cut may convert to lowdust
# 2. smoothing
LOWDUST_SMOOTH_FWHM = 5.0                # deg; 0 = no smoothing
# 3. hole filling
HOLE_FILL_LABELS = ["dusty_plane", "nes", "scp"]   # regions that can absorb lowdust holes
HOLE_MODE = "component"                  # "component": whole hole -> one region; "pixel"
# 4. bulgy islands
BULGY_ISLAND_MAX_AREA = 60.0             # deg^2; bulgy pieces smaller than this are removed
BULGY_TRIM_FRAC = 0.10                   # then trim this fraction of bulgy area off its edge (0 = off)
# 5. bridge (Galactic box, degrees; l may wrap through 0)
BRIDGE_MODE = "ecliptic"                 # "ecliptic": band along the ecliptic; "box": Galactic box
BRIDGE_TARGET_AREA = 400.0               # ecliptic mode: band width is solved to give this area (deg^2)
BRIDGE_MAX_WIDTH = 30.0                  # ecliptic mode: never wider than this (deg), so the band hugs the ecliptic
BRIDGE_ELAT_OFFSET = 0.0                 # ecliptic mode: centre the band at this ecliptic latitude
BRIDGE_L = (-16.0, 16.0)                 # box mode: l range, as (min, max) in [-180, 180]
BRIDGE_B = (-45.0, -5.0)                 # box mode: b range
BRIDGE_TAKE_FROM = ["dusty_plane"]       # only these labels become bridge
BRIDGE_LABEL = "bridge"
BRIDGE_IN_WFD = True                     # count the bridge as WFD in the totals
# 6. Galactic-plane WFD region: keep scheduler "bulgy", or replace it with the
#    Roman Galactic Plane Survey (RGPS) wide-field footprint grown to RGPS_TARGET_AREA
GALPLANE_MODE = "rgps_tvs"               # "bulgy", "rgps" (grown Roman), "rgps_tvs" (TVS-shaped)
RGPS_TARGET_AREA = 1700.0                # deg^2 after growing the RGPS border
RGPS_LABEL = "rgps"
RGPS_PROTECT = ["lowdust", "LMC_SMC", "euclid_overlap", "virgo"]   # growth never takes these
RGPS_DEC_MAX = 15.0                      # growth stays south of this Dec (scheduler plane limit)
RGPS_TVS_INCLUDE_BRIDGE = False          # rgps_tvs: False = TVS ecliptic-bridge pixels stay bridge, not rgps
BRIDGE_SEED_TVS = True                   # rgps_tvs: the bridge always includes the TVS ecliptic-bridge pixels
RGPS_NORTH_L_MIN = 30.0                  # rgps_tvs: northern limit (deg). 30 keeps the wide TVS plane
                                         # (|b| out to ~6-11 deg) and passes the TVS bridge (reaches l ~ 28);
                                         # beyond l ~ 30 the TVS region is a thin |b| < ~3 deg strip
RGPS_EDGE_TRIM_FRAC = 0.0                # peel this fraction of rgps area off its border, evenly all round (0 = off)
RGPS_EDGE_TRIM_TO_AREA = None            # or: peel the border until rgps is this area (deg^2); overrides the fraction
RGPS_SMOOTH_FWHM = 4.0                   # smooth the rgps border with a Gaussian of this FWHM (deg); 0 = off
RGPS_SINGLE_PIECE = True                 # no islands: keep only the largest connected piece of rgps
RGPS_MIN_NEIGHBOURS = 3                  # cleanup: drop rgps pixels with fewer rgps neighbours (of 8) than this,
                                         # and fill notches with at least 8 - this + 2 (=7) rgps neighbours
RGPS_FILL_HOLES_MAX = 60.0               # fill enclosed non-WFD holes inside rgps smaller than this (deg^2)
RGPS_NORTH_CUT = True                    # rgps_tvs: drop ALL rgps at l > RGPS_NORTH_L_MIN (thin northern plane)
RGPS_GROW_TO_TARGET = False              # rgps_tvs: if short of RGPS_TARGET_AREA, grow back to it (else leave short)
BRIDGE_ABSORB_RGPS_SHORTFALL = True      # add rgps shortfall (RGPS_TARGET_AREA - rgps) to the bridge target
BRIDGE_NORTH_PLANE_B = 6.0               # with RGPS_NORTH_CUT: bridge never takes l > RGPS_NORTH_L_MIN, |b| < this
# RGPS_BOXES (the Roman wide-field footprint) is defined in section 10a

npix = hp.nside2npix(NSIDE)
NBRS = hp.get_all_neighbours(NSIDE, np.arange(npix))      # (8, npix), -1 = none


def to_frame_nearest(m, src, dst):
    """Rotate by nearest-pixel lookup (no interpolation): good for masks and labels."""
    if src == dst:
        return m
    th, ph = hp.pix2ang(NSIDE, np.arange(m.size))
    th2, ph2 = hp.Rotator(coord=[dst, src])(th, ph)
    return m[hp.ang2pix(NSIDE, th2, ph2)]


def components(mask):
    """Connected groups (8-neighbour) of True pixels: returns (pixels, group id per pixel)."""
    pix = np.where(mask)[0]
    idx = -np.ones(npix, dtype=int)
    idx[pix] = np.arange(pix.size)
    nb = NBRS[:, pix]
    ok = (nb >= 0) & mask[np.clip(nb, 0, None)]
    r = np.broadcast_to(np.arange(pix.size), nb.shape)[ok]
    adj = coo_matrix((np.ones(r.size), (r, idx[nb[ok]])), shape=(pix.size, pix.size))
    return pix, connected_components(adj, directed=False)[1]


def fill_nearest(labels, mask, cand_labels, mode="component"):
    """Relabel the pixels in mask with the nearest region in cand_labels.
    Returns (pixels, assigned labels, group ids)."""
    cand = np.isin(labels, cand_labels) & ~mask
    if not mask.any():
        return np.array([], int), np.array([], "U20"), np.array([], int)
    if not cand.any():
        raise ValueError(f"no pixels carry any of {cand_labels}")
    cpix = np.where(cand)[0]
    tree = cKDTree(np.array(hp.pix2vec(NSIDE, cpix)).T)
    if mode == "pixel":
        pix = np.where(mask)[0]
        _, j = tree.query(np.array(hp.pix2vec(NSIDE, pix)).T)
        out, gid = labels[cpix[j]], np.arange(pix.size)
    else:
        pix, gid = components(mask)
        out = np.empty(pix.size, dtype="U20")
        for g in np.unique(gid):
            members = pix[gid == g]
            touch = NBRS[:, members].ravel()
            touch = touch[(touch >= 0)]
            touch = touch[cand[touch]]
            if touch.size:            # touches candidate regions: most shared edges
                labs, cnt = np.unique(labels[touch], return_counts=True)
                out[gid == g] = labs[np.argmax(cnt)]
            else:                     # enclosed: closest candidate pixel
                d, j = tree.query(np.array(hp.pix2vec(NSIDE, members)).T)
                out[gid == g] = labels[cpix[j[np.argmin(d)]]]
    labels[pix] = out
    return pix, out, gid


def report(title, out, gid, cand_labels):
    for l in cand_labels:
        sel_ = out == l
        if sel_.any():
            print(f"    {title} -> {l:14s} {sel_.sum() * pix_area:8,.0f} deg^2"
                  f"  ({np.unique(gid[sel_]).size} pieces)")


old_labels = np.asarray(fp_labels).astype("U20")
new_labels = old_labels.copy()

# ---- 1. cut ----
cut_c = (side_mask(to_frame_nearest(logdens, star_frame, "C"), np.log10(STARDENS_CUT), STARDENS_SIDE)
         & side_mask(to_frame_nearest(ebv, dust_frame, "C"), EBV_CUT, EBV_SIDE))
is_low = old_labels == "lowdust"
claim = np.isin(old_labels, CLAIM_FROM_LABELS) & cut_c
low = (is_low & cut_c) | claim
print(f"1. cut:     lowdust kept {(is_low & cut_c).sum() * pix_area:,.0f} deg^2, "
      f"failed {(is_low & ~cut_c).sum() * pix_area:,.0f} deg^2, "
      f"claimed {claim.sum() * pix_area:,.0f} deg^2 (from {CLAIM_FROM_LABELS or 'none'})")

# ---- 2. smooth the lowdust edge (normalised convolution) ----
tradeable = np.isin(old_labels, HOLE_FILL_LABELS) | is_low | claim
if LOWDUST_SMOOTH_FWHM > 0:
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        num = hp.smoothing(low.astype(float), fwhm=np.radians(LOWDUST_SMOOTH_FWHM))
        den = hp.smoothing(tradeable.astype(float), fwhm=np.radians(LOWDUST_SMOOTH_FWHM))
    frac = np.where(den > 0.05, num / np.maximum(den, 1e-6), 0.0)
    smooth_low = tradeable & (frac >= 0.5)
    gained = smooth_low & ~low & np.isin(old_labels, HOLE_FILL_LABELS + ["lowdust"])
    lost = low & ~smooth_low
    low = (low & ~lost) | gained
    print(f"2. smooth:  FWHM {LOWDUST_SMOOTH_FWHM:g} deg, lowdust gained {gained.sum() * pix_area:,.0f} deg^2, "
          f"lost {lost.sum() * pix_area:,.0f} deg^2")
else:
    print("2. smooth:  off")

# ---- 3. fill holes left in the old lowdust area ----
new_labels[low] = "lowdust"
holes = is_low & ~low
new_labels[holes] = ""                   # temporarily unassigned
pix, out, gid = fill_nearest(new_labels, holes, HOLE_FILL_LABELS, HOLE_MODE)
print(f"3. holes:   {holes.sum() * pix_area:,.0f} deg^2 in {np.unique(gid).size} pieces")
report("hole", out, gid, HOLE_FILL_LABELS)

if GALPLANE_MODE == "bulgy":
    # ---- 4. squash small bulgy islands ----
    bpix, bgid = components(new_labels == "bulgy")
    sizes = np.bincount(bgid) * pix_area if bpix.size else np.array([])
    small = np.isin(bgid, np.where(sizes < BULGY_ISLAND_MAX_AREA)[0])
    islands = np.zeros(npix, bool)
    islands[bpix[small]] = True
    print(f"4. bulgy:   {sizes.size} pieces, sizes (deg^2): "
          + ", ".join(f"{a:,.0f}" for a in sorted(sizes, reverse=True)))
    if islands.any():
        new_labels[islands] = ""
        _, out, gid = fill_nearest(new_labels, islands, HOLE_FILL_LABELS, "component")
        print(f"           removed {islands.sum() * pix_area:,.0f} deg^2 in {np.unique(gid).size} islands "
              f"< {BULGY_ISLAND_MAX_AREA:g} deg^2")
        report("island", out, gid, HOLE_FILL_LABELS)

    # ---- 4b. trim the bulgy edge by BULGY_TRIM_FRAC of its area ----
    if BULGY_TRIM_FRAC > 0:
        bul = np.where(new_labels == "bulgy")[0]
        toward = np.where(np.isin(new_labels, HOLE_FILL_LABELS))[0]
        n_trim = int(round(BULGY_TRIM_FRAC * bul.size))
        if n_trim and toward.size:
            # distance from each bulgy pixel to the nearest pixel it can be trimmed into;
            # edges shared with lowdust / other WFD regions are not trimmed
            d, _ = cKDTree(np.array(hp.pix2vec(NSIDE, toward)).T).query(np.array(hp.pix2vec(NSIDE, bul)).T)
            trim = np.zeros(npix, bool)
            trim[bul[np.argsort(d, kind="stable")[:n_trim]]] = True
            new_labels[trim] = ""
            _, out, gid = fill_nearest(new_labels, trim, HOLE_FILL_LABELS, "pixel")
            a0 = (old_labels == "bulgy").sum() * pix_area
            a1 = (new_labels == "bulgy").sum() * pix_area
            print(f"4b. trim:   removed {trim.sum() * pix_area:,.0f} deg^2 from the bulgy edge "
                  f"({100 * BULGY_TRIM_FRAC:g}% of {bul.size * pix_area:,.0f}); bulgy now {a1:,.0f} deg^2 "
                  f"({100 * (a1 / a0 - 1):+.1f}% vs current footprint)")
            for l in HOLE_FILL_LABELS:
                if (out == l).any():
                    print(f"    trim -> {l:14s} {(out == l).sum() * pix_area:8,.0f} deg^2")
            n_pieces = np.unique(components(new_labels == "bulgy")[1]).size
            print(f"           bulgy now in {n_pieces} pieces")

# ---- 4c. replace bulgy with the grown RGPS footprint ----
th, ph = hp.pix2ang(NSIDE, np.arange(npix))
gth, gph = hp.Rotator(coord=["C", "G"])(th, ph)
gl = (np.degrees(gph) + 180.0) % 360.0 - 180.0
gb = 90.0 - np.degrees(gth)
dec_all = 90.0 - np.degrees(th)
PLANE = "bulgy"
core = np.zeros(npix, bool)
for name, ((l0, l1), (b0, b1)) in RGPS_BOXES.items():
    core |= (gl >= l0) & (gl < l1) & (gb >= b0) & (gb < b1)
n_target = int(round(RGPS_TARGET_AREA / pix_area))
gc_vec = np.array(hp.ang2vec(*hp.Rotator(coord=["G", "C"])(np.pi / 2, 0.0)))   # Galactic centre
dist_gc = np.degrees(np.arccos(np.clip(np.array(hp.pix2vec(NSIDE, np.arange(npix))).T @ gc_vec, -1, 1)))
if GALPLANE_MODE == "rgps_tvs":
    PLANE = RGPS_LABEL
    shape = tvs_region if RGPS_TVS_INCLUDE_BRIDGE else tvs_gp
    # TVS ecliptic-bridge pixels outside the TVS plane region are reserved for the bridge
    reserved = (tvs_bridge & ~tvs_gp) if not RGPS_TVS_INCLUDE_BRIDGE else np.zeros(npix, bool)
    cand = shape & ~np.isin(new_labels, RGPS_PROTECT) & ~reserved
    if RGPS_NORTH_CUT:
        north_cut = cand & (gl > RGPS_NORTH_L_MIN)
        cand &= ~north_cut
    print(f"4c. RGPS (TVS-shaped): TVS shape {shape.sum() * pix_area:,.0f} deg^2"
          + ("" if RGPS_TVS_INCLUDE_BRIDGE else " (without its ecliptic bridge)")
          + f"; {(shape & np.isin(new_labels, RGPS_PROTECT)).sum() * pix_area:,.0f} deg^2 of it in protected regions, skipped"
          + (f"; TVS ecliptic bridge reserved for the bridge: {reserved.sum() * pix_area:,.0f} deg^2" if reserved.any() else ""))
    rgps = cand.copy()
    if cand.sum() > n_target:
        # trim: northern plane first (l > RGPS_NORTH_L_MIN, farthest l first), then
        # anything else farthest from the Galactic centre
        cpix = np.where(cand)[0]
        north = gl[cpix] > RGPS_NORTH_L_MIN
        key = np.where(north, -1e3 - gl[cpix], -dist_gc[cpix])     # most negative = removed first
        drop = cpix[np.argsort(key, kind="stable")[:cand.sum() - n_target]]
        rgps[drop] = False
        dropped = np.zeros(npix, bool)
        dropped[drop] = True
        print(f"           trimmed {dropped.sum() * pix_area:,.0f} deg^2: "
              f"{(dropped & (gl > RGPS_NORTH_L_MIN)).sum() * pix_area:,.0f} from the northern plane "
              f"(now l <= {gl[rgps & (gl > 0)].max() if (rgps & (gl > 0)).any() else 0:.1f} deg on that side), "
              f"{(dropped & (gl <= RGPS_NORTH_L_MIN)).sum() * pix_area:,.0f} elsewhere")
    elif cand.sum() < n_target and RGPS_GROW_TO_TARGET:
        # grow: nearest ring of pixels first, and within a ring those closest to the Galactic centre
        eligible = ~cand & ~reserved & ~np.isin(new_labels, RGPS_PROTECT) & (dec_all <= RGPS_DEC_MAX)
        epix = np.where(eligible)[0]
        d, _ = cKDTree(np.array(hp.pix2vec(NSIDE, np.where(cand)[0])).T).query(
            np.array(hp.pix2vec(NSIDE, epix)).T)
        ring = np.floor(np.degrees(d) / hp.nside2resol(NSIDE, arcmin=True) * 60.0)
        add = epix[np.lexsort((dist_gc[epix], ring))[:n_target - cand.sum()]]
        rgps[add] = True
        print(f"           grown by {add.size * pix_area:,.0f} deg^2 next to the TVS shape, nearest the bulge first")
    if RGPS_NORTH_CUT:
        print(f"           northern thin plane removed (l > {RGPS_NORTH_L_MIN:g} deg): "
              f"{north_cut.sum() * pix_area:,.0f} deg^2 of the TVS shape")
        l_br = gl[tvs_bridge].max()
        # TVS plane width just inside the cut, to confirm the wide part is kept
        band_in = tvs_gp & (gl > RGPS_NORTH_L_MIN - 4) & (gl <= RGPS_NORTH_L_MIN)
        band_out = tvs_gp & (gl > RGPS_NORTH_L_MIN) & (gl <= RGPS_NORTH_L_MIN + 4)
        w_in = band_in.sum() * pix_area / 4 if band_in.any() else 0.0
        w_out = band_out.sum() * pix_area / 4 if band_out.any() else 0.0
        print(f"           TVS bridge reaches l = {l_br:.1f} deg "
              + ("(cut is north of it: OK)" if RGPS_NORTH_L_MIN >= l_br else "<-- cut is SOUTH of the bridge; raise RGPS_NORTH_L_MIN"))
        print(f"           TVS plane width in b: {w_in:.1f} deg just inside the cut, {w_out:.1f} deg just outside")
    if cand.sum() < n_target and not RGPS_GROW_TO_TARGET:
        print(f"           rgps left {n_target * pix_area - rgps.sum() * pix_area:,.0f} deg^2 short of "
              f"{RGPS_TARGET_AREA:g} (RGPS_GROW_TO_TARGET = False)")
    taken = {l or "(outside footprint)": (rgps & (new_labels == l)).sum() * pix_area
             for l in np.unique(new_labels[rgps])}
    new_labels[rgps] = RGPS_LABEL
    print(f"           rgps = {rgps.sum() * pix_area:,.0f} deg^2; overlap with TVS region "
          f"{(rgps & tvs_region).sum() * pix_area:,.0f} deg^2; Roman wide-field kept "
          f"{(rgps & core).sum() * pix_area:,.0f} of {core.sum() * pix_area:,.0f} deg^2")
    print("           taken from: " + ", ".join(f"{k} {v:,.0f}" for k, v in taken.items()))
    left = new_labels == "bulgy"
    if left.any():
        new_labels[left] = ""
        _, out, gid = fill_nearest(new_labels, left, HOLE_FILL_LABELS, "component")
        print(f"           remaining bulgy {left.sum() * pix_area:,.0f} deg^2 reassigned:")
        report("bulgy", out, gid, HOLE_FILL_LABELS)
elif GALPLANE_MODE == "rgps":
    PLANE = RGPS_LABEL
    print(f"4c. RGPS:   core footprint {core.sum() * pix_area:,.0f} deg^2 at nside {NSIDE} "
          f"(691 deg^2 nominal)")
    over = core & np.isin(new_labels, RGPS_PROTECT)
    if over.any():
        print(f"           core overlaps protected regions by {over.sum() * pix_area:,.0f} deg^2 (core kept)")
    # grow: add the eligible pixels closest to the core until the target area is reached
    eligible = ~core & ~np.isin(new_labels, RGPS_PROTECT) & (dec_all <= RGPS_DEC_MAX)
    n_add = max(n_target - core.sum(), 0)
    epix = np.where(eligible)[0]
    d, _ = cKDTree(np.array(hp.pix2vec(NSIDE, np.where(core)[0])).T).query(
        np.array(hp.pix2vec(NSIDE, epix)).T)
    grown = np.zeros(npix, bool)
    grown[epix[np.argsort(d, kind="stable")[:n_add]]] = True
    rgps = core | grown
    buffer_deg = np.degrees(2 * np.arcsin(np.sort(d)[n_add - 1] / 2)) if n_add else 0.0
    taken = {l or "(outside footprint)": (rgps & (new_labels == l)).sum() * pix_area
             for l in np.unique(new_labels[rgps])}
    new_labels[rgps] = RGPS_LABEL
    print(f"           grown by up to {buffer_deg:.1f} deg to {rgps.sum() * pix_area:,.0f} deg^2; taken from: "
          + ", ".join(f"{k} {v:,.0f}" for k, v in taken.items()))
    # whatever is left of bulgy goes to the nearest non-WFD region
    left = new_labels == "bulgy"
    if left.any():
        new_labels[left] = ""
        _, out, gid = fill_nearest(new_labels, left, HOLE_FILL_LABELS, "component")
        print(f"           remaining bulgy {left.sum() * pix_area:,.0f} deg^2 reassigned:")
        report("bulgy", out, gid, HOLE_FILL_LABELS)

# ---- 4c-2. trim the rgps border evenly (pixels nearest the edge go first) ----
edge_trimmed = np.zeros(npix, bool)          # trimmed area: dusty_plane, not counted toward the bridge
if GALPLANE_MODE != "bulgy" and (RGPS_EDGE_TRIM_FRAC > 0 or RGPS_EDGE_TRIM_TO_AREA is not None):
    rg = new_labels == RGPS_LABEL
    if RGPS_EDGE_TRIM_TO_AREA is not None:
        n_trim = max(rg.sum() - int(round(RGPS_EDGE_TRIM_TO_AREA / pix_area)), 0)
    else:
        n_trim = int(round(RGPS_EDGE_TRIM_FRAC * rg.sum()))
    if n_trim:
        rpix = np.where(rg)[0]
        d, _ = cKDTree(np.array(hp.pix2vec(NSIDE, np.where(~rg)[0])).T).query(
            np.array(hp.pix2vec(NSIDE, rpix)).T)
        trim = np.zeros(npix, bool)
        trim[rpix[np.argsort(d, kind="stable")[:n_trim]]] = True
        depth = np.degrees(2 * np.arcsin(np.sort(d)[n_trim - 1] / 2))
        a0 = rg.sum() * pix_area
        new_labels[trim] = "dusty_plane"
        edge_trimmed = trim
        print(f"4c-2. edge: trimmed {trim.sum() * pix_area:,.0f} deg^2 off the rgps border "
              f"(up to {depth:.1f} deg in): {a0:,.0f} -> {(new_labels == RGPS_LABEL).sum() * pix_area:,.0f} deg^2; trimmed area -> dusty_plane")

# ---- 4d. smooth the rgps border (area-preserving) ----
if GALPLANE_MODE != "bulgy" and RGPS_SMOOTH_FWHM > 0:
    rg = new_labels == RGPS_LABEL
    rgps_before_smooth = rg.copy()
    # pixels the border may move into: non-WFD regions, minus the reserved TVS bridge and
    # the northern cut (so smoothing never refills them)
    allowed = (rg | np.isin(new_labels, HOLE_FILL_LABELS)) & ~edge_trimmed
    if GALPLANE_MODE == "rgps_tvs":
        allowed &= ~reserved
        if RGPS_NORTH_CUT:
            allowed &= ~(gl > RGPS_NORTH_L_MIN)
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        num = hp.smoothing(rg.astype(float), fwhm=np.radians(RGPS_SMOOTH_FWHM))
        den = hp.smoothing(allowed.astype(float), fwhm=np.radians(RGPS_SMOOTH_FWHM))
    frac = np.where(allowed & (den > 0.05), num / np.maximum(den, 1e-6), -1.0)
    # keep the same number of pixels: the n highest smoothed values (threshold ~0.5)
    n_keep = rg.sum()
    keep = np.zeros(npix, bool)
    keep[np.argsort(-frac, kind="stable")[:n_keep]] = True
    gained, lost = keep & ~rg, rg & ~keep
    new_labels[gained] = RGPS_LABEL
    if lost.any():
        new_labels[lost] = ""
        fill_nearest(new_labels, lost, HOLE_FILL_LABELS, "pixel")
    specks = np.zeros(npix, bool)
    print(f"4d. smooth: rgps border smoothed (FWHM {RGPS_SMOOTH_FWHM:g} deg): moved {gained.sum() * pix_area:,.0f} deg^2 "
          f"in / {lost.sum() * pix_area:,.0f} out" + (f", removed {specks.sum() * pix_area:,.0f} deg^2 of specks" if specks.any() else "")
          + f"; rgps = {(new_labels == RGPS_LABEL).sum() * pix_area:,.0f} deg^2 in {np.unique(components(new_labels == RGPS_LABEL)[1]).size} pieces")

# ---- 4e. no islands: one connected rgps piece, small enclosed holes filled ----
if GALPLANE_MODE != "bulgy" and RGPS_SINGLE_PIECE and (new_labels == RGPS_LABEL).any():
    # pixel-scale cleanup: shave pixels hanging on by a corner, fill one-pixel notches
    shaved = notched = 0
    for _ in range(10):
        rg_ = new_labels == RGPS_LABEL
        nbs = NBRS.copy()
        n_rg = np.where(nbs >= 0, rg_[np.clip(nbs, 0, None)], False).sum(axis=0)
        thin = rg_ & (n_rg < RGPS_MIN_NEIGHBOURS)
        notch = (~rg_ & np.isin(new_labels, HOLE_FILL_LABELS) & ~edge_trimmed & (n_rg >= 10 - RGPS_MIN_NEIGHBOURS))
        if GALPLANE_MODE == "rgps_tvs":
            notch &= ~reserved
            if RGPS_NORTH_CUT:
                notch &= ~(gl > RGPS_NORTH_L_MIN)
        if not (thin.any() or notch.any()):
            break
        new_labels[notch] = RGPS_LABEL
        notched += notch.sum()
        if thin.any():
            new_labels[thin] = ""
            fill_nearest(new_labels, thin, HOLE_FILL_LABELS, "pixel")
            shaved += thin.sum()
    rpix, rgid = components(new_labels == RGPS_LABEL)
    sizes = np.bincount(rgid) * pix_area
    islands = np.zeros(npix, bool)
    islands[rpix[rgid != np.argmax(sizes)]] = True
    if islands.any():
        new_labels[islands] = ""
        fill_nearest(new_labels, islands, HOLE_FILL_LABELS, "component")
    # enclosed holes: connected non-rgps pieces (non-WFD labels) whose neighbours are all rgps
    fillable = np.isin(new_labels, HOLE_FILL_LABELS)
    filled = 0.0
    if fillable.any() and RGPS_FILL_HOLES_MAX > 0:
        hpix, hgid = components(fillable)
        rg_now = new_labels == RGPS_LABEL
        for g in np.unique(hgid):
            mem = hpix[hgid == g]
            if mem.size * pix_area >= RGPS_FILL_HOLES_MAX:
                continue
            nb = NBRS[:, mem].ravel()
            nb = nb[nb >= 0]
            nb = nb[~np.isin(nb, mem)]
            if nb.size and rg_now[nb].all():
                new_labels[mem] = RGPS_LABEL
                filled += mem.size * pix_area
    n_pieces = np.unique(components(new_labels == RGPS_LABEL)[1]).size
    print(f"4e. islands: removed {islands.sum() * pix_area:,.0f} deg^2 in {sizes.size - 1} detached piece(s)"
          + (" (sizes " + ", ".join(f"{a:,.0f}" for a in sorted(np.delete(sizes, np.argmax(sizes)), reverse=True)) + " deg^2)" if sizes.size > 1 else "")
          + f"; filled {filled:,.0f} deg^2 of enclosed holes; shaved {shaved * pix_area:,.0f} / notches filled "
          f"{notched * pix_area:,.0f} deg^2; rgps = {(new_labels == RGPS_LABEL).sum() * pix_area:,.0f} deg^2 in {n_pieces} piece")

# close-up of the plane in Galactic coordinates: rgps border before / after smoothing
if GALPLANE_MODE != "bulgy" and RGPS_SMOOTH_FWHM > 0:
    figz = plt.figure(figsize=(15, 4.2))
    for k, (m, ttl) in enumerate([(rgps_before_smooth, "rgps before smoothing"),
                                  (new_labels == RGPS_LABEL, "rgps after smoothing + no islands")]):
        hp.cartview(m.astype(float), fig=figz.number, sub=(1, 2, k + 1), coord=["C", "G"],
                    lonra=[-95, 45], latra=[-30, 30], cmap="Oranges", min=0, max=1.6,
                    cbar=False, title=ttl, notext=True)
        hp.graticule(dpar=10, dmer=20, alpha=0.4)
    plt.show()

# ---- 5. bridge between the plane region and lowdust ----
low_mask = new_labels == "lowdust"
takeable = np.isin(new_labels, BRIDGE_TAKE_FROM) & ~edge_trimmed
if GALPLANE_MODE == "rgps_tvs" and RGPS_NORTH_CUT:
    # keep the bridge out of the northern plane that was just removed from rgps
    north_plane = (gl > RGPS_NORTH_L_MIN) & (np.abs(gb) < BRIDGE_NORTH_PLANE_B)
    if RGPS_NORTH_CUT:
        north_plane |= north_cut
    takeable &= ~north_plane
plane_mask = new_labels == PLANE


def touches(mask, other):
    nb = NBRS[:, np.where(mask)[0]].ravel()
    return other[nb[nb >= 0]].any()


bridge_target = BRIDGE_TARGET_AREA
if BRIDGE_ABSORB_RGPS_SHORTFALL and GALPLANE_MODE != "bulgy":
    shortfall = max(RGPS_TARGET_AREA - edge_trimmed.sum() * pix_area
                    - (new_labels == RGPS_LABEL).sum() * pix_area, 0.0)
    bridge_target += shortfall
    if shortfall > 0:
        print(f"5. bridge:  target {BRIDGE_TARGET_AREA:g} + rgps shortfall {shortfall:,.0f} = {bridge_target:,.0f} deg^2")
bridge_seed = np.zeros(npix, bool)
if GALPLANE_MODE == "rgps_tvs" and BRIDGE_SEED_TVS:
    bridge_seed = tvs_bridge & ~tvs_gp & takeable
    print(f"5. bridge:  seeded with {bridge_seed.sum() * pix_area:,.0f} deg^2 of TVS ecliptic-bridge pixels")
if BRIDGE_MODE == "ecliptic":
    eth, _ = hp.Rotator(coord=["C", "E"])(th, ph)
    elat = 90.0 - np.degrees(eth)

    def ecl_bridge(width):
        # takeable pixels within width/2 of the ecliptic, keeping only the
        # connected pieces that touch the plane region (the bulge crossing)
        band = takeable & ((np.abs(elat - BRIDGE_ELAT_OFFSET) <= width / 2) | bridge_seed)
        if not band.any():
            return band
        bpix_, bgid_ = components(band)
        out_ = np.zeros(npix, bool)
        for g in np.unique(bgid_):
            m = np.zeros(npix, bool)
            m[bpix_[bgid_ == g]] = True
            if touches(m, plane_mask):
                out_ |= m
        return out_

    lo, hi = 0.0, BRIDGE_MAX_WIDTH
    if ecl_bridge(hi).sum() * pix_area < bridge_target:
        width = hi
        print(f"5. bridge:  at the maximum width ({hi:g} deg) the band gives only "
              f"{ecl_bridge(hi).sum() * pix_area:,.0f} of {bridge_target:,.0f} deg^2; "
              "raise BRIDGE_MAX_WIDTH or accept the shortfall")
    else:
        for _ in range(30):                     # bisection on the band width
            mid = 0.5 * (lo + hi)
            if ecl_bridge(mid).sum() * pix_area < bridge_target:
                lo = mid
            else:
                hi = mid
        # pick whichever bracketing width lands closer to the target
        a_lo, a_hi = ecl_bridge(lo).sum() * pix_area, ecl_bridge(hi).sum() * pix_area
        width = lo if abs(a_lo - bridge_target) <= abs(a_hi - bridge_target) else hi
    bridge = ecl_bridge(width)
    if ecl_bridge(0.0).sum() * pix_area > bridge_target:
        print(f"           note: the TVS bridge seed alone exceeds the bridge target ({bridge_target:,.0f} deg^2)")
    desc = f"ecliptic band {width:.2f} deg wide (|elat - {BRIDGE_ELAT_OFFSET:g}| <= {width / 2:.2f})"
else:
    in_box = (gl >= BRIDGE_L[0]) & (gl <= BRIDGE_L[1]) & (gb >= BRIDGE_B[0]) & (gb <= BRIDGE_B[1])
    bridge = (in_box & takeable) | bridge_seed
    desc = f"box l={BRIDGE_L}, b={BRIDGE_B}"
new_labels[bridge] = BRIDGE_LABEL
print(f"5. bridge:  {bridge.sum() * pix_area:,.0f} deg^2 relabelled {BRIDGE_LABEL!r} ({desc})")
for side, sel_b in [("north (b > 0)", gb > 0), ("south (b < 0)", gb < 0)]:
    n = (bridge & sel_b).sum() * pix_area
    print(f"           bridge {side}: {n:,.0f} deg^2")

# connectivity check: is the plane region joined to lowdust on each side of the plane?
j_pix, j_gid = components(np.isin(new_labels, [PLANE, "lowdust", BRIDGE_LABEL]))
gid_of = np.full(npix, -1)
gid_of[j_pix] = j_gid
bul = np.where(new_labels == PLANE)[0]
main_bulge = np.bincount(gid_of[bul]).argmax() if bul.size else -1
# lowdust pixels that the plane region reaches only through the bridge
reach = (gid_of == main_bulge)
plane_bridge = np.isin(new_labels, [PLANE, BRIDGE_LABEL])
adj_low = np.zeros(npix, bool)
nb = NBRS[:, np.where(plane_bridge & reach)[0]].ravel()
adj_low[nb[nb >= 0]] = True
adj_low &= new_labels == "lowdust"
for side, sel_b in [("north (b > 0)", gb > 0), ("south (b < 0)", gb < 0)]:
    ok = (adj_low & sel_b).any()
    print(f"           {PLANE} connected to lowdust {side}: {'YES' if ok else 'NO'}")


### 10b. Area and pointings per component

In [ ]:
# Area and pointings per component
comps = [l for l in ["lowdust", "euclid_overlap", "virgo", "LMC_SMC", "bulgy", RGPS_LABEL, BRIDGE_LABEL,
                     "dusty_plane", "nes", "scp"]
         if l in set(old_labels) | set(new_labels)]
comps += sorted((set(old_labels) | set(new_labels)) - set(comps) - {""})
rows = []
for c in comps:
    a_old = (old_labels == c).sum() * pix_area
    a_new = (new_labels == c).sum() * pix_area
    rows.append({"component": c, "WFD": c in WFD_LABELS or c == RGPS_LABEL or (c == BRIDGE_LABEL and BRIDGE_IN_WFD),
                 "area_current_deg2": a_old, "area_new_deg2": a_new,
                 "change_deg2": a_new - a_old,
                 "pointings_current": a_old * POINTINGS_PER_DEG2,
                 "pointings_new": a_new * POINTINGS_PER_DEG2})
base = pd.DataFrame(rows)
num = base.drop(columns=["component", "WFD"])
totals = pd.DataFrame([
    {"component": "WFD total", "WFD": True, **num[base.WFD].sum().to_dict()},
    {"component": "non-WFD total", "WFD": False, **num[~base.WFD].sum().to_dict()},
    {"component": "footprint total", "WFD": None, **num.sum().to_dict()},
])
tab = pd.concat([base, totals], ignore_index=True)

tab.to_csv(NEW_FP_FILE.replace(".fits", "_areas.csv"), index=False)
display(tab.style.format({c: "{:,.0f}" for c in tab.columns if c not in ("component", "WFD")})
        .hide(axis="index"))

### 10c. Visit budget

Total visits per component = pointings × median visits per pointing, for both the current and the modified footprint. `bridge` gets the WFD value unless set separately. For scale, `rubin_scheduler` budgets about 2.2 million visits for a 10-year survey (`Footprint.estimate_counts`).

In [ ]:
WFD_VISITS = 750
VISITS = {
    "lowdust": WFD_VISITS, "bulgy": WFD_VISITS, "LMC_SMC": WFD_VISITS,
    "euclid_overlap": WFD_VISITS, "virgo": WFD_VISITS,
    BRIDGE_LABEL: WFD_VISITS, RGPS_LABEL: WFD_VISITS,
    "dusty_plane": 170, "nes": 150, "scp": 100,
}

vt = base[["component", "WFD", "pointings_current", "pointings_new"]].copy()
vt["median_visits"] = vt.component.map(VISITS)
missing = vt.loc[vt.median_visits.isna(), "component"].tolist()
if missing:
    print("no visit count set for:", missing, "(counted as 0)")
vt["median_visits"] = vt.median_visits.fillna(0)
vt["visits_current"] = vt.pointings_current * vt.median_visits
vt["visits_new"] = vt.pointings_new * vt.median_visits
vt["change"] = vt.visits_new - vt.visits_current
cols = ["pointings_current", "pointings_new", "visits_current", "visits_new", "change"]
vt = pd.concat([vt, pd.DataFrame([
    {"component": "WFD total", "WFD": True, **vt.loc[vt.WFD, cols].sum().to_dict()},
    {"component": "non-WFD total", "WFD": False, **vt.loc[~vt.WFD, cols].sum().to_dict()},
    {"component": "TOTAL", "WFD": None, **vt[cols].sum().to_dict()},
])], ignore_index=True)

vt.to_csv(NEW_FP_FILE.replace(".fits", "_visits.csv"), index=False)
display(vt.style.format({c: "{:,.0f}" for c in cols + ["median_visits"]}, na_rep="")
        .hide(axis="index"))
tot = vt.iloc[-1]
print(f"total visits: current footprint {tot.visits_current:,.0f}, modified footprint {tot.visits_new:,.0f} "
      f"({tot.change:+,.0f})")

In [ ]:
# Map of the modified footprint, plus a FITS file of the new labels
order = [c for c in comps if (new_labels == c).any()]
code = np.full(new_labels.size, hp.UNSEEN)
for i, c in enumerate(order):
    code[new_labels == c] = i
code_p = to_frame_nearest(code, "C", target)

palette = ["#2a9d8f", "#e9c46a", "#f4a261", "#8ab17d", "#e76f51",
           "#6d597a", "#457b9d", "#a8dadc", "#b5838d", "#ffb4a2"]
colors = (palette * 2)[:len(order)]

fig4 = plt.figure(figsize=(11, 6.5))
hp.mollview(code_p, fig=fig4.number, cmap=ListedColormap(colors), min=-0.5,
            max=len(order) - 0.5, cbar=False,
            title=f"Modified footprint: joint cut replaces lowdust [{frame}]")
hp.graticule(dpar=30, dmer=60, alpha=0.3)
outline(to_frame_nearest(old_labels == "lowdust", "C", target), colors=["k"],
        linewidths=0.8, linestyles=":")
handles = [Patch(color=col, label=f"{c}: {(new_labels == c).sum() * pix_area:,.0f} deg$^2$")
           for c, col in zip(order, colors)]
handles.append(Line2D([], [], color="k", ls=":", lw=0.8, label="current lowdust"))
if GALPLANE_MODE in ("rgps", "rgps_tvs"):
    outline(to_frame_nearest(core, "C", target), colors=["#222222"], linewidths=1.0)
    handles.append(Line2D([], [], color="#222222", lw=1.0, label="RGPS wide-field (Roman)"))

# DECaPS2 footprint outline: CDS HiPS MOC if reachable, else the published box
SHOW_DECAPS = True
DECAPS_SOURCE = "moc"          # "moc" (download coverage map, falls back to box) or "box"
DECAPS_MOC_URL = "https://alasky.cds.unistra.fr/DECaPS/DR2/CDS_P_DECaPS_DR2_g/Moc.fits"
DECAPS_BOX = ((-124.0, 6.0), (-10.0, 10.0))   # (l range), (b range) deg; Saydjari et al. 2023
DECAPS_COLOR = "#00b4d8"


def moc_to_mask(path_or_url, nside):
    """Rasterise a MOC FITS (NUNIQ column) onto a RING map at nside (any-overlap)."""
    from astropy.io import fits as _fits
    with _fits.open(path_or_url, cache=True) as f:
        uniq = np.asarray(f[1].data.field(0), dtype=np.int64)
    order = (np.floor(np.log2(uniq // 4) / 2)).astype(int)
    ipix = uniq - 4 * 4 ** order.astype(np.int64)
    o_out = int(np.log2(nside))
    nest = []
    hi = order >= o_out
    nest.append(ipix[hi] >> (2 * (order[hi] - o_out)))          # finer cells -> parent
    for o in np.unique(order[~hi]):                             # coarser cells -> children
        p = ipix[order == o]
        k = 4 ** (o_out - o)
        nest.append((p[:, None] * k + np.arange(k)[None, :]).ravel())
    m = np.zeros(hp.nside2npix(nside), bool)
    m[hp.nest2ring(nside, np.unique(np.concatenate(nest)))] = True
    return m


if SHOW_DECAPS:
    decaps = None
    if DECAPS_SOURCE == "moc":
        try:
            decaps = moc_to_mask(DECAPS_MOC_URL, NSIDE)
            decaps_src = "CDS HiPS MOC (DR2)"
        except Exception as e:
            print(f"DECaPS MOC not available ({type(e).__name__}); using the published box")
    if decaps is None:
        (dl0, dl1), (db0, db1) = DECAPS_BOX
        decaps = (gl >= dl0) & (gl <= dl1) & (gb >= db0) & (gb <= db1)
        decaps_src = f"box {dl1:g} > l > {dl0:g}, {db0:g} < b < {db1:g}"
    print(f"DECaPS2 ({decaps_src}): {decaps.sum() * pix_area:,.0f} deg^2; overlap by region: "
          + ", ".join(f"{l} {(decaps & (new_labels == l)).sum() * pix_area:,.0f}"
                      for l in order if (decaps & (new_labels == l)).any()))
    outline(to_frame_nearest(decaps, "C", target), colors=[DECAPS_COLOR], linewidths=1.4,
            linestyles="--")
    handles.append(Line2D([], [], color=DECAPS_COLOR, lw=1.4, ls="--", label="DECaPS2"))

# TVS SC proposed Galactic-plane region (built in section 10c-2)
SHOW_TVS_GP = True
TVS_GP_COLOR = "#ff006e"
if SHOW_TVS_GP:
    print("TVS region overlap with modified footprint: " + ", ".join(
        f"{l} {(tvs_region & (new_labels == l)).sum() * pix_area:,.0f}" for l in order
        if (tvs_region & (new_labels == l)).any()) + " deg^2")
    outline(to_frame_nearest(tvs_region, "C", target), colors=[TVS_GP_COLOR], linewidths=1.6,
            linestyles="-.")
    handles.append(Line2D([], [], color=TVS_GP_COLOR, lw=1.6, ls="-.",
                          label=f"TVS SC proposed GP region ({tvs_region.sum() * pix_area:,.0f} deg$^2$)"))

# Ecliptic (lat 0, solid) plus optional parallel lines of ecliptic latitude (dashed)
ECLIPTIC_LATS = [0.0]          # e.g. [-10.0, 0.0, 10.0] to show the NES +/-10 deg band
ECLIPTIC_COLOR = "#ffd23f"
elon = np.linspace(0.0, 360.0, 721)
for elat in ECLIPTIC_LATS:
    eth, eph = np.radians(90.0 - elat) * np.ones_like(elon), np.radians(elon)
    if target != "E":
        eth, eph = hp.Rotator(coord=["E", target])(eth, eph)
    hp.projplot(eth, eph,
                color=ECLIPTIC_COLOR, lw=1.6 if elat == 0 else 1.0,
                ls="-" if elat == 0 else "--")
handles.append(Line2D([], [], color=ECLIPTIC_COLOR, lw=1.6, label="ecliptic"))
if any(l != 0 for l in ECLIPTIC_LATS):
    handles.append(Line2D([], [], color=ECLIPTIC_COLOR, lw=1.0, ls="--",
                          label="ecliptic lat " + ", ".join(f"{l:+g}" for l in ECLIPTIC_LATS if l != 0) + " deg"))
plt.gca().legend(handles=handles, loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8, framealpha=0.9)
fig4.savefig(NEW_FP_FILE.replace(".fits", ".png"), dpi=150, bbox_inches="tight")
plt.show()

### 10d. Export the modified footprint as HEALPix maps

Writes the modified footprint in two forms. Both are equatorial and RING-ordered, at `OUT_NSIDE`. The default is 64, the working resolution; the scheduler's default is 32.

- **`footprint_GPmod_nside{N}.fits`:** a multi-column HEALPix FITS file with these columns:
  - `label`: an integer region code, −1 outside the footprint. The code-to-name key is in the header as `LABEL0`, `LABEL1`, …
  - `wfd`: 1 for WFD regions, 0 otherwise.
  - `median_visits`: the per-region median visits from section 10c.
  - `u`, `g`, `r`, `i`, `z`, `y`: the scheduler's per-band target weights, from the band ratios each region has in the current `rubin_scheduler` footprint.
- **`footprint_GPmod_nside{N}.npz`:** the same footprint in the shape `get_current_footprint()` returns. That is a structured array `footprints` with fields u…y, plus a string array `labels`, so it can be handed straight to scheduler or MAF code.

The new regions borrow the band ratios of an existing one: `rgps` uses `bulgy`'s and `bridge` uses `lowdust`'s by default (`BORROW_RATIOS`). Changing to a different `OUT_NSIDE` uses nearest-pixel lookup, so labels are never blended.

In [ ]:
OUT_NSIDE = 64                     # 64 = working resolution; 32 = rubin_scheduler default
OUT_STEM = f"footprint_GPmod_nside{OUT_NSIDE}"
BORROW_RATIOS = {RGPS_LABEL: "bulgy", BRIDGE_LABEL: "lowdust"}   # new region -> ratios of
BANDS = ["u", "g", "r", "i", "z", "y"]

# Per-band ratios for each region, read from the current scheduler footprint
fp_maps, fp_lab = get_current_footprint(nside=NSIDE)
ratios = {}
for lab in np.unique(fp_lab[fp_lab != ""]):
    ratios[lab] = {b: float(np.median(fp_maps[b][fp_lab == lab])) for b in BANDS}
for new, src in BORROW_RATIOS.items():
    ratios[new] = ratios[src]
missing = sorted(set(order) - set(ratios))
if missing:
    raise ValueError(f"no band ratios for {missing}; add them to BORROW_RATIOS")
print(pd.DataFrame(ratios).T.loc[order].round(3).to_string())

# Resample the label map to OUT_NSIDE by nearest pixel (labels are never blended)
if OUT_NSIDE == NSIDE:
    out_labels = new_labels
else:
    th_o, ph_o = hp.pix2ang(OUT_NSIDE, np.arange(hp.nside2npix(OUT_NSIDE)))
    out_labels = new_labels[hp.ang2pix(NSIDE, th_o, ph_o)]
n_out = out_labels.size

code_of = {c: i for i, c in enumerate(order)}
label_col = np.array([code_of.get(l, -1) for l in out_labels], dtype=np.int32)
wfd_col = np.array([(l in WFD_LABELS) or l == RGPS_LABEL or (l == BRIDGE_LABEL and BRIDGE_IN_WFD)
                    for l in out_labels], dtype=np.int16)
visits_col = np.array([VISITS.get(l, 0) if l else 0 for l in out_labels], dtype=np.float32)
footprints = np.zeros(n_out, dtype=list(zip(BANDS, [float] * len(BANDS))))
for l in order:
    m = out_labels == l
    for b in BANDS:
        footprints[b][m] = ratios[l][b]

# 1) multi-column HEALPix FITS
cols = [label_col, wfd_col, visits_col] + [footprints[b].astype(np.float32) for b in BANDS]
names = ["label", "wfd", "median_visits"] + BANDS
hdr = [(f"LABEL{i}", c) for i, c in enumerate(order)] + [
    ("STARMOD", STAR_MODEL), ("FILTER", FILTER), ("MAGLIM", MAG_LIMIT),
    ("SDCUT", STARDENS_CUT), ("SDSIDE", STARDENS_SIDE), ("EBVCUT", EBV_CUT), ("EBVSIDE", EBV_SIDE),
    ("GPMODE", GALPLANE_MODE), ("RGPSAREA", RGPS_TARGET_AREA if GALPLANE_MODE != "bulgy" else 0.0),
    ("BRMODE", BRIDGE_MODE), ("BRAREA", float(bridge_target)),
]
hp.write_map(OUT_STEM + ".fits", cols, coord="C", column_names=names,
             dtype=[np.int32, np.int16, np.float32] + [np.float32] * len(BANDS),
             extra_header=hdr, overwrite=True)

# 2) scheduler-style npz: same layout as get_current_footprint()
np.savez(OUT_STEM + ".npz", footprints=footprints, labels=out_labels.astype("U20"))

# Read-back check
chk = hp.read_map(OUT_STEM + ".fits", field=None)
pa_out = hp.nside2pixarea(OUT_NSIDE, degrees=True)
print(f"wrote {OUT_STEM}.fits ({len(names)} columns: {', '.join(names)}) and {OUT_STEM}.npz")
print(f"nside {OUT_NSIDE}: footprint {(chk[0] >= 0).sum() * pa_out:,.0f} deg^2, "
      f"WFD {(chk[1] == 1).sum() * pa_out:,.0f} deg^2")

fig5 = plt.figure(figsize=(16, 5))
hp.mollview(np.where(label_col >= 0, visits_col, hp.UNSEEN), fig=fig5.number, sub=(1, 2, 1),
            cmap="viridis", title="median visits per pixel (exported)", unit="visits")
hp.graticule(dpar=30, dmer=60, alpha=0.3)
hp.mollview(np.where(label_col >= 0, footprints["r"], hp.UNSEEN), fig=fig5.number, sub=(1, 2, 2),
            cmap="magma", title="r-band target weight (exported)", unit="relative to lowdust r")
hp.graticule(dpar=30, dmer=60, alpha=0.3)
plt.show()